In [ ]:
# ============================================
# SPAIN RETAIL BIG DATA PIPELINE - PYSPARK
# ============================================

# ============================================
# 1. INSTALL & IMPORTS
# ============================================

!pip install pyspark==3.5.1

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ============================================
# 2. START SPARK SESSION
# ============================================

spark = SparkSession.builder \
    .appName("Spain_Retail_BigData_Pipeline") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print("Spark Version:", spark.version)

In [2]:
# ============================================
# 3. DEFINE MANUAL SCHEMA
# ============================================

# Read InvoiceDate as String first (safer for CSV)
# Read UnitPrice as Double first, then cast later to Decimal

manual_schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Country", StringType(), True)
])

# ============================================
# 4. READ CSV FILE
# ============================================

df_raw = spark.read.csv(
    "Online_Retail.csv",
    header=True,
    schema=manual_schema
)

print(f"\nTotal Records All Countries: {df_raw.count():,}")

print("\n=== RAW DATA SAMPLE ===")
df_raw.show(5, truncate=False)

print("\n=== RAW SCHEMA ===")
df_raw.printSchema()


Total Records All Countries: 541,909

=== RAW DATA SAMPLE ===
+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate   |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/2010 8:26|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |12/1/2010 8:26|3.39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |12/1/2010 8:26|2.75     |17850     |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |12/1/2010 8:26|3.39     |17850     |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |12/1/2010 8:26|3.39     |17850     |United Ki

In [3]:
# ============================================
# 5. CONVERT DATE COLUMN
# ============================================

df_raw = df_raw.withColumn(
    "InvoiceDate",
    to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")
)

# ============================================
# 6. FILTER SPAIN ONLY
# ============================================

df_spain_raw = df_raw.filter(col("Country") == "Spain")

print(f"\nTotal Spain Records: {df_spain_raw.count():,}")


Total Spain Records: 2,533


In [4]:

# ============================================
# 7. DATA CLEANING & FEATURE ENGINEERING
# ============================================

df_clean = df_spain_raw \
    .filter(col("CustomerID").isNotNull()) \
    .filter(col("Quantity") > 0) \
    .filter(col("UnitPrice") > 0) \
    .filter(col("Description").isNotNull()) \
    .filter(col("InvoiceDate").isNotNull()) \
    .filter(~col("InvoiceNo").startswith("C")) \
    .dropDuplicates(["InvoiceNo", "StockCode", "CustomerID"]) \
    .withColumn(
        "UnitPrice",
        col("UnitPrice").cast(DecimalType(10,2))
    ) \
    .withColumn(
        "TotalAmount",
        round(col("Quantity") * col("UnitPrice"), 2)
    ) \
    .withColumn(
        "YearMonth",
        date_format("InvoiceDate", "yyyy-MM")
    ) \
    .withColumn(
        "InvoiceYear",
        year("InvoiceDate")
    ) \
    .withColumn(
        "InvoiceMonth",
        month("InvoiceDate")
    )

print(f"\nRecords after cleaning: {df_clean.count():,}")

print(
    f"Data Quality Loss: "
    f"{(1 - df_clean.count()/df_spain_raw.count())*100:.1f}%"
)

print("\n=== CLEAN DATA SAMPLE ===")
df_clean.show(5, truncate=False)


Records after cleaning: 2,463
Data Quality Loss: 2.8%

=== CLEAN DATA SAMPLE ===
+---------+---------+-----------------------+--------+-------------------+---------+----------+-------+-----------+---------+-----------+------------+
|InvoiceNo|StockCode|Description            |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country|TotalAmount|YearMonth|InvoiceYear|InvoiceMonth|
+---------+---------+-----------------------+--------+-------------------+---------+----------+-------+-----------+---------+-----------+------------+
|536944   |20725    |LUNCH BAG RED RETROSPOT|70      |2010-12-03 12:20:00|1.65     |12557     |Spain  |115.50     |2010-12  |2010       |12          |
|536944   |20727    |LUNCH BAG  BLACK SKULL.|60      |2010-12-03 12:20:00|1.65     |12557     |Spain  |99.00      |2010-12  |2010       |12          |
|536944   |20728    |LUNCH BAG CARS BLUE    |100     |2010-12-03 12:20:00|1.45     |12557     |Spain  |145.00     |2010-12  |2010       |12          |
|536944   |2

In [ ]:
# ============================================
# 8. BUILD STAR SCHEMA
# ============================================

# ---------- DIM CUSTOMERS ----------

dim_customers = df_clean.select(
    "CustomerID",
    "Country"
).distinct() \
.withColumn(
    "CustomerKey",
    monotonically_increasing_id()
)

# ---------- DIM PRODUCTS ----------

dim_products = df_clean.select(
    "StockCode",
    "Description",
    "UnitPrice"
) \
.groupBy("StockCode", "Description") \
.agg(
    round(avg("UnitPrice"), 2).alias("AvgUnitPrice_EUR")
) \
.withColumn(
    "ProductKey",
    monotonically_increasing_id()
)

# ---------- DIM DATE ----------

dim_date = df_clean.select("InvoiceDate") \
.distinct() \
.withColumn(
    "DateKey",
    date_format("InvoiceDate", "yyyyMMdd").cast("int")
) \
.withColumn(
    "Year",
    year("InvoiceDate")
) \
.withColumn(
    "Month",
    month("InvoiceDate")
) \
.withColumn(
    "Day",
    dayofmonth("InvoiceDate")
) \
.withColumn(
    "Quarter",
    quarter("InvoiceDate")
)

# ---------- FACT ORDERS ----------

fact_orders = df_clean.select(
    "InvoiceNo",
    "CustomerID",
    "InvoiceDate"
).distinct() \
.withColumn(
    "DateKey",
    date_format("InvoiceDate", "yyyyMMdd").cast("int")
)

# ---------- FACT TRANSACTIONS ----------

fact_transactions = df_clean.select(
    "InvoiceNo",
    "StockCode",
    "Quantity",
    "UnitPrice",
    "TotalAmount"
)

# ============================================
# 9. CACHE DIMENSIONS
# ============================================

dim_customers.cache()
dim_products.cache()
dim_date.cache()

In [6]:
# ============================================
# 10. KPI 1 - MONTHLY REVENUE TREND
# ============================================

monthly_revenue = df_clean.groupBy("YearMonth") \
    .agg(
        round(sum("TotalAmount"), 2).alias("Revenue_EUR"),
        countDistinct("InvoiceNo").alias("TotalOrders"),
        countDistinct("CustomerID").alias("ActiveCustomers")
    ) \
    .orderBy("YearMonth")

window_spec = Window.orderBy("YearMonth")

monthly_revenue_trend = monthly_revenue \
    .withColumn(
        "PrevMonthRevenue",
        lag("Revenue_EUR", 1).over(window_spec)
    ) \
    .withColumn(
        "MoM_Growth_%",
        round(
            (
                col("Revenue_EUR") - col("PrevMonthRevenue")
            ) / col("PrevMonthRevenue") * 100,
            2
        )
    )

print("\n=== MONTHLY REVENUE TREND - SPAIN ===")

monthly_revenue_trend.show(truncate=False)


=== MONTHLY REVENUE TREND - SPAIN ===
+---------+-----------+-----------+---------------+----------------+------------+
|YearMonth|Revenue_EUR|TotalOrders|ActiveCustomers|PrevMonthRevenue|MoM_Growth_%|
+---------+-----------+-----------+---------------+----------------+------------+
|2010-12  |1843.73    |4          |4              |NULL            |NULL        |
|2011-01  |10086.09   |8          |7              |1843.73         |447.05      |
|2011-02  |2114.50    |4          |4              |10086.09        |-79.04      |
|2011-03  |5363.15    |8          |8              |2114.50         |153.64      |
|2011-04  |1785.65    |3          |3              |5363.15         |-66.71      |
|2011-05  |3257.60    |4          |3              |1785.65         |82.43       |
|2011-06  |3333.21    |8          |8              |3257.60         |2.32        |
|2011-07  |7606.27    |9          |7              |3333.21         |128.20      |
|2011-08  |3308.99    |10         |6              |7606.27 

In [7]:
# ============================================
# 11. KPI 2 - TOP PRODUCTS
# ============================================

top_products_spain = fact_transactions \
    .join(
        broadcast(dim_products),
        "StockCode"
    ) \
    .groupBy("StockCode", "Description") \
    .agg(
        round(sum("TotalAmount"), 2).alias("Revenue_EUR"),
        sum("Quantity").alias("UnitsSold")
    ) \
    .orderBy(col("Revenue_EUR").desc()) \
    .limit(10)

print("\n=== TOP 10 PRODUCTS - SPAIN ===")

top_products_spain.show(truncate=False)


=== TOP 10 PRODUCTS - SPAIN ===
+---------+---------------------------------+-----------+---------+
|StockCode|Description                      |Revenue_EUR|UnitsSold|
+---------+---------------------------------+-----------+---------+
|POST     |POSTAGE                          |5852.00    |209      |
|84997D   |PINK 3 PIECE POLKADOT CUTLERY SET|3957.75    |1089     |
|84997D   |CHILDRENS CUTLERY POLKADOT PINK  |3957.75    |1089     |
|84997C   |CHILDRENS CUTLERY POLKADOT BLUE  |3671.15    |1013     |
|84997C   |BLUE 3 PIECE POLKADOT CUTLERY SET|3671.15    |1013     |
|22423    |REGENCY CAKESTAND 3 TIER         |2049.00    |172      |
|84997B   |RED 3 PIECE RETROSPOT CUTLERY SET|1044.76    |292      |
|84997B   |CHILDRENS CUTLERY RETROSPOT RED  |1044.76    |292      |
|20728    |LUNCH BAG CARS BLUE              |810.70     |558      |
|84997A   |CHILDRENS CUTLERY POLKADOT GREEN |774.76     |220      |
+---------+---------------------------------+-----------+---------+



In [8]:
# ============================================
# 12. KPI 3 - RFM SEGMENTATION
# ============================================

max_date = fact_orders \
    .agg(max("InvoiceDate")) \
    .collect()[0][0]

rfm_base = fact_transactions \
    .join(fact_orders, "InvoiceNo") \
    .groupBy("CustomerID") \
    .agg(
        max("InvoiceDate").alias("LastPurchaseDate"),
        countDistinct("InvoiceNo").alias("Frequency"),
        round(sum("TotalAmount"), 2).alias("Monetary_EUR")
    )

rfm_spain = rfm_base.withColumn(
    "Recency",
    datediff(lit(max_date), col("LastPurchaseDate"))
)

rfm_window_r = Window.orderBy(col("Recency").asc())
rfm_window_f = Window.orderBy(col("Frequency").desc())
rfm_window_m = Window.orderBy(col("Monetary_EUR").desc())

rfm_final = rfm_spain \
    .withColumn(
        "R_Score",
        ntile(5).over(rfm_window_r)
    ) \
    .withColumn(
        "F_Score",
        ntile(5).over(rfm_window_f)
    ) \
    .withColumn(
        "M_Score",
        ntile(5).over(rfm_window_m)
    ) \
    .withColumn(
        "RFM_Segment",
        when(
            (col("R_Score") >= 4) &
            (col("F_Score") >= 4),
            "Champions"
        )
        .when(
            (col("R_Score") >= 3) &
            (col("F_Score") >= 3),
            "Loyal Customers"
        )
        .when(
            col("M_Score") >= 4,
            "Big Spenders"
        )
        .when(
            (col("R_Score") <= 2) &
            (col("F_Score") <= 2),
            "At Risk"
        )
        .when(
            col("R_Score") <= 2,
            "Can't Lose Them"
        )
        .otherwise("Needs Attention")
    )

print("\n=== RFM CUSTOMER SEGMENTATION ===")

rfm_final.groupBy("RFM_Segment") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()


=== RFM CUSTOMER SEGMENTATION ===
+---------------+-----+
|    RFM_Segment|count|
+---------------+-----+
|        At Risk|    9|
|Loyal Customers|    9|
|      Champions|    6|
|Needs Attention|    3|
|   Big Spenders|    3|
+---------------+-----+



In [9]:
 # ============================================
# 13. SAVE TO PARQUET
# ============================================

output_path = "/content/spain_retail_parquet"

df_clean.write.mode("overwrite") \
    .parquet(f"{output_path}/fact_transactions")

dim_customers.write.mode("overwrite") \
    .parquet(f"{output_path}/dim_customers")

dim_products.write.mode("overwrite") \
    .parquet(f"{output_path}/dim_products")

rfm_final.write.mode("overwrite") \
    .parquet(f"{output_path}/rfm_segments")

print(f"\n=== DATA SAVED TO PARQUET ===")
print(output_path)


=== DATA SAVED TO PARQUET ===
/content/spain_retail_parquet


In [10]:
# ============================================
# 14. FINAL VALIDATION
# ============================================

print("\n=== SPAIN MARKET VALIDATION ===")

total_revenue = df_clean \
    .agg(sum("TotalAmount")) \
    .collect()[0][0]

total_customers = dim_customers.count()

avg_order_value = df_clean \
    .groupBy("InvoiceNo") \
    .agg(sum("TotalAmount").alias("OrderValue")) \
    .agg(avg("OrderValue")) \
    .collect()[0][0]

print(f"Total Revenue Spain: {total_revenue:,.2f} EUR")
print(f"Total Spanish Customers: {total_customers:,}")
print(f"Average Order Value: {avg_order_value:,.2f} EUR")
print(f"Total Clean Transactions: {df_clean.count():,}")

print(
    f"Date Range: "
    f"{df_clean.agg(min('InvoiceDate'), max('InvoiceDate')).collect()[0]}"
)


=== SPAIN MARKET VALIDATION ===
Total Revenue Spain: 61,452.20 EUR
Total Spanish Customers: 30
Average Order Value: 682.80 EUR
Total Clean Transactions: 2,463
Date Range: Row(min(InvoiceDate)=datetime.datetime(2010, 12, 3, 12, 20), max(InvoiceDate)=datetime.datetime(2011, 12, 7, 17, 5))


In [11]:
# ============================================
# 15. STOP SPARK SESSION
# ============================================

spark.stop()